In [ ]:
#install dependencies
%pip install google-genai 
%pip install neo4j neo4j-graphrag
%pip install dotenv
%pip install langchain_google_genai

In [ ]:
from google import genai
from google.genai import types
from neo4j import GraphDatabase
from dotenv import load_dotenv
import os

# Load environment variables from .env file
load_dotenv('dx.env', override=True)

# --- 1. SETTINGS & CREDENTIALS ---
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
NEO4J_URI = os.getenv("NEO4J_URI")
NEO4J_USER = os.getenv("NEO4J_USERNAME")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD")
NEO4J_DATABASE = os.getenv("NEO4J_DATABASE_NAME")
NEO4J_VECTOR_INDEX = os.getenv("NEO4J_INDEX_NAME")

NEO4J_AUTH = (NEO4J_USER, NEO4J_PASSWORD)
TOP_K = 5

In [ ]:

client = genai.Client(api_key=GEMINI_API_KEY)

# 2. Call models.embed_content
result = client.models.embed_content(
    model="gemini-embedding-001",
    contents="chest pain",
    config=types.EmbedContentConfig(
        task_type="RETRIEVAL_QUERY"
    )
)

# 3. Access the embedding values
print(result.embeddings[0].values)

In [ ]:


# --- 1. GENERATE GEMINI QUERY EMBEDDING ---
ai_client = genai.Client(api_key=GEMINI_API_KEY)

# --- 2. Retrieve embeddings for search query 'diabetes'
embedding_response = ai_client.models.embed_content(
    model="gemini-embedding-001",
    contents="chest pain",
    config=types.EmbedContentConfig(
        task_type="RETRIEVAL_QUERY"
    )
)

queryVector = [float(val) for val in embedding_response.embeddings[0].values]

# --- 3. EXECUTE NEO4J VECTOR SEARCH ---
# Built in Function
cypher_query = """
CALL db.index.vector.queryNodes($index_name, $k, $queryVector)
YIELD node, score
RETURN node.text AS text, score
LIMIT $k
"""

# CYPHER 25 SEARCH Clause - GQL Aligned
cypher25_query = """
CYPHER 25
MATCH (chunk:__Chunk__)
SEARCH chunk IN (
  VECTOR INDEX vector_index___Chunk___embedding FOR $queryVector
  LIMIT $k
) SCORE AS score
RETURN chunk.text AS text, score
"""

with GraphDatabase.driver(NEO4J_URI, auth=NEO4J_AUTH) as driver:
    records, summary, keys = driver.execute_query(
        cypher25_query,
        index_name=NEO4J_VECTOR_INDEX,
        k=TOP_K,
        queryVector=queryVector,
        database_=NEO4J_DATABASE
    )

   # --- 4. PRINT RESULTS ---
    for record in records:
        print(f"Score: {record['score']:.4f}")
        content_str = str({record['text']})
        preview = content_str[:200] + '...' if len(content_str) > 200 else content_str
        print(f"Text: {preview}")
        #print(f"Score: {record['score']:.4f} | Text: {record['text']}")

In [ ]:
#Query Embeddings and Relationships

# --- 1. Retrieve embeddings for search query 'diabetes'
embedding_response = ai_client.models.embed_content(
    model="gemini-embedding-001",
    contents="diabetes",
    config=types.EmbedContentConfig(
        task_type="RETRIEVAL_QUERY"
    )
)

queryVector = [float(val) for val in embedding_response.embeddings[0].values]

# --- 2. Query for embeddings where Chunks related to Claims that have been Denies
# Query includes chunks and their relationships to Claims nodes
cypher25_query = """
CYPHER 25
MATCH (chunk:__Chunk__)
SEARCH chunk IN (
  VECTOR INDEX vector_index___Chunk___embedding FOR $queryVector
  LIMIT $k
) SCORE AS score
MATCH (claim:Claim)-[:__NODE_TO_CHUNK__]->(chunk)
WHERE claim.status = 'Denied'
RETURN score, claim.claimId AS claimId, claim.status AS status
"""


with GraphDatabase.driver(NEO4J_URI, auth=NEO4J_AUTH) as driver:
    records, summary, keys = driver.execute_query(
        cypher25_query,
        index_name=NEO4J_VECTOR_INDEX,
        k=TOP_K,
        queryVector=queryVector,
        database_=NEO4J_DATABASE
    )

# --- 3. Print Results
    for record in records:
        print(f"Score: {record['score']:.4f} | ClaimID: {record['claimId']} | ClaimStatus: {record['status']}")
       # print(f"Score: {record['score']:.4f} | Text: {record['text']}")

## Vector Retriever

The following cel introduces the `VectorRetriever` for semantic search and connects it to an LLM to build a complete GraphRAG question-answering pipeline.

**Learning Objectives:**
- Use `VectorRetriever` to find semantically similar chunks
- Build a `GraphRAG` pipeline that retrieves context and generates answers
- Inspect the context passed to the LLM

The retrieval pipeline works in three steps: the query is embedded, the vector index returns the most similar chunks, and those chunks are passed to the LLM as context for generating an answer.

In [ ]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings
# define an embedder function for our queries
def get_embedder(task_type: str = "retrieval_query"):
    """
    Returns an operational Gemini embedding object.
    Supported task_types: "retrieval_document", "retrieval_query", "semantic_similarity"
    """
    # The SDK automatically checks for os.environ["GOOGLE_API_KEY"]
    return GoogleGenerativeAIEmbeddings(
        model="models/gemini-embedding-001",
        task_type=task_type
    )

## Initialize Vector Retriever
The `VectorRetriever` wraps the vector index and handles embedding the query, running the similarity search, and formatting the results. The `return_properties` parameter controls which node properties are included in the retrieval output.

In [ ]:
from neo4j_graphrag.retrievers import VectorRetriever
from neo4j_graphrag.generation import GraphRAG

#initialize a VectorRetriever

embedder = get_embedder() #note Google API requires parameter

driver = GraphDatabase.driver(
    NEO4J_URI,
    auth=(NEO4J_USER, NEO4J_PASSWORD),
    notifications_min_severity='OFF',
)
driver.verify_connectivity()

vector_retriever = VectorRetriever(
    driver=driver,
    index_name=NEO4J_VECTOR_INDEX,
    embedder=embedder,
    return_properties=['text']
)

print('Vector retriever initialized!')

## Search for Relevant Chunks
Use the retriever to find the 5 chunks most semantically similar to a query. Each result includes a similarity score and the chunk text.

In [ ]:
#test our retriever witth a natural language query
query = "Which patients have diabetes?"
result = vector_retriever.search(query_text=query, top_k=5)

print(f'Query: "{query}"')
print(f'Results returned: {len(result.items)}\n')

for i, item in enumerate(result.items, 1):
    score = item.metadata.get('score', 0.0)
    content_preview = str(item.content)[:150]
    print(f'{i}. Score: {score:.4f}')
    print(f'   {content_preview}...\n')

## Build GraphRAG Pipeline
The `GraphRAG` class connects the retriever to the LLM. When you call `search()`, it retrieves relevant chunks, assembles them into a context prompt, and sends that prompt to the LLM for answer generation.

Setting `return_context=True` lets you inspect the exact chunks the LLM received.

In [ ]:
def get_llm():
    # Instantiate and return the neo4j_graphrag GeminiLLM wrapper
    return GeminiLLM(
        model_name="gemini-3.6-flash",
        model_params={"temperature": 0.0},
        # api_key=os.getenv("GEMINI_API_KEY") # optional if set in env vars
    )


In [ ]:
import logging
from neo4j_graphrag.generation import GraphRAG

# Suppress the google-genai SDK warning log - comes from neo4j_graphrage framework
logging.getLogger("google.genai").setLevel(logging.ERROR)


# What get_llm() safely instantiates for you under the hood:
from neo4j_graphrag.llm import GeminiLLM  # For google-genai
# OR 
#from neo4j_graphrag.llm import VertexAILLM # For vertexai

# Then it passes seamlessly into your workshop tasks:
llm = get_llm() 

rag = GraphRAG(llm=llm, retriever=vector_retriever)

query = "What are risk codes associated with these patients?"
response = rag.search(query, retriever_config={'top_k': 5}, return_context=True)

print(f'Query: "{query}"')
print(f'Chunks retrieved: {len(response.retriever_result.items)}\n')
print('Answer:')
print(response.answer)

print('\n\n=== Retrieved Context ===')
for i, item in enumerate(response.retriever_result.items, 1):
    score = item.metadata.get('score', 0.0)
    content_str = str(item.content)
    preview = content_str[:200] + '...' if len(content_str) > 200 else content_str
    print(f'\n[{i}] Score: {score:.4f}')
    print(f'    {preview}')

## Try Different Queries
Experiment with different questions to see how semantic search finds relevant content even when the exact words differ from the source text.

In [ ]:
query = "How many patients have diabetes?"

response = rag.search(query, retriever_config={'top_k': 3})

print(f'Query: "{query}"\n')
print('Answer:')
print(response.answer)

## VectorCypher Retriever
The `VectorCypherRetriever` enhances vector search with custom Cypher graph traversal. After the vector index finds matching chunks, a Cypher query runs on each match to pull in related graph context: neighboring chunks, document metadata, or connected entities.

**Learning Objectives:**
- Write a Cypher retrieval query that traverses from matched chunks to related nodes
- Use VectorCypherRetriever to combine semantic search with graph context
- Compare results between pure vector and vector-cypher approaches

Pure vector search returns isolated chunks. VectorCypher returns the chunk plus whatever the graph knows about its neighborhood.

### Define Retrieval Query
The retrieval query runs on each chunk returned by vector search. It receives the matched node and score variables and must return columns that the result formatter will process.

This query traverses from the matched chunk to its associated `Claim`, then follows the `HAS_CLAIM` relationship to find the Patient that filed it. 

In [ ]:

NEO4J_RETRIEVAL_QUERY = """
MATCH (patient:Patient)-[:HAS_CLAIM]->(claim:Claim)-[:__NODE_TO_CHUNK__]->(node)
WITH node, score, patient, claim
RETURN node.text AS text,
       score,
       { claimID: claim.claimId,
         claimStatus: claim.status,
         patientName: patient.name
       } AS metadata
"""


print('Retrieval query defined!')

## Initialize VectorCypher Retriever
The `VectorCypherRetriever` takes the same vector index and embedder as the basic `VectorRetriever`, plus the custom Cypher query that will run on each match.

A custom `result_formatter` separates the chunk text (passed to the LLM as context) from the structured metadata (available for programmatic inspection). Without a formatter, the default serializes the entire record into a single string.

In [ ]:

import neo4j
from neo4j_graphrag.retrievers import VectorCypherRetriever
from neo4j_graphrag.types import RetrieverResultItem

def format_record(record: neo4j.Record) -> RetrieverResultItem:
    """Separate chunk text (content for LLM) from structured graph metadata."""
    metadata = record.get("metadata") or {}
    metadata["score"] = record.get("score")
    return RetrieverResultItem(
        content=record.get("text", ""),
        metadata=metadata,
    )

#update retriever to use query and formatter
vector_cypher_retriever = VectorCypherRetriever(
    driver=driver,
    index_name=NEO4J_VECTOR_INDEX,
    embedder=embedder,
    retrieval_query=NEO4J_RETRIEVAL_QUERY,
    result_formatter=format_record,
)

print('VectorCypherRetriever initialized!')

## Search with Graph Context
Run a query with the `VectorCypherRetriever` and compare the enriched results to the plain chunk text you saw from the `VectorRetriever` previously. Notice the additional metadata — claimId, claimStatus, patientName — that the graph traversal adds to each match.

In [ ]:
query = "Which patients have diabetes?"

cypher_result = vector_cypher_retriever.search(query_text=query, top_k=3)

print('=== VectorCypherRetriever ===')
for i, item in enumerate(cypher_result.items, 1):
    meta = item.metadata or {}
    print(f'\n[{i}] Score: {meta.get("score", 0):.4f}')
    print(f'    ClaimId: {meta.get("claimId", "N/A")}')
    print(f'    ClaimStatus: {meta.get("claimStatus", "N/A")}')
    print(f'    PatientName: {meta.get("patientName", "N/A")}')
    content_preview = str(item.content)[:200]
    print(f'    Text: {content_preview}...')

## GraphRAG with Graph Context
Build a GraphRAG pipeline with the VectorCypher retriever. The LLM now receives the matched chunk along with connected entity data — companies, products, and risk factors — producing answers grounded in both unstructured text and structured knowledge graph relationships.

In [ ]:
rag = GraphRAG(llm=llm, retriever=vector_cypher_retriever)

query = "What are the primary illnesses mentioned in the doctor's notes?"
response = rag.search(query, retriever_config={'top_k': 5}, return_context=True)

print(f'Query: "{query}"\n')
print('Answer:')
print(response.answer)

print('\n\n=== Enriched Context ===')
for i, item in enumerate(response.retriever_result.items, 1):
    meta = item.metadata or {}
    print(f'\n[{i}] Score: {meta.get("score", 0):.4f} |  ClaimId: {meta.get("claimId", "N/A")}')
    print(f'    ClaimStatus: {meta.get("claimStatus", "N/A")} | PatientName: {meta.get("patientName", "N/A")}')
    content_preview = str(item.content)[:200]
    print(f'    Text: {content_preview}...')



## Summary

The `VectorCypherRetriever` adds graph context to every vector search result:

| Approach | What the LLM Receives |
|----------|----------------------|
| **VectorRetriever** | The matched chunk text only |
| **VectorCypherRetriever** | The matched chunk text + filing metadata + connected companies, products, and risk factors |

The retrieval query gathers Claims and Patients linked to the matched chunk via `__NODE_TO_CHUNK__`, keeping the context scoped to what the chunk actually mentions. A custom `result_formatter` separates the chunk text (passed to the LLM) from structured metadata (available for inspection).

Both approaches rely on vector similarity to find the initial matches. The graph traversal enriches those matches with structured knowledge — this is the core of GraphRAG.
